# SaaS / Subscription Churn Prediction & Analysis

## Exploratory Data Analysis

### Objective

Explore the raw customer subscription dataset to understand:

- Dataset structure and data quality
- Customer and subscription characteristics
- Churn distribution
- Numerical feature distributions
- Categorical feature distributions
- Potential relationships between customer attributes and churn

### Business Objective

Identify customers at risk of churn and quantify the recurring revenue potentially at risk so that the business can prioritize retention efforts.

### Analysis Principle

This analysis will prioritize business relevance, interpretability, and clear reasoning over unnecessary feature complexity.

## EDA Roadmap

1. Load the raw dataset
2. Inspect dataset structure
3. Audit data quality
4. Investigate potential hidden/malformed missing values
5. Analyze churn class balance
6. Analyze numerical variables
7. Analyze categorical variables
8. Compare customer characteristics by churn status
9. Identify initial business patterns
10. Document observations and implications

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

## 1. Load Raw Dataset

The raw dataset is loaded directly from `data/raw/` and is not modified during the initial inspection stage.

In [22]:
file_path = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(file_path)

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Dataset Structure

The first structural checks establish the dataset size, column names, data types, and overall composition before any cleaning or transformation is performed.

In [23]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

Rows: 7,043
Columns: 21


In [24]:
df.columns.tolist()

['customerID',
 'gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges',
 'Churn']

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [26]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customerID,7043,7043,7590-VHVEG,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,7043,2,Male,3555,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SeniorCitizen,"7,043.00",NaN,NaN,NaN,0.16,0.37,0.00,0.00,0.00,0.00,1.00
Partner,7043,2,No,3641,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dependents,7043,2,No,4933,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure,"7,043.00",NaN,NaN,NaN,32.37,24.56,0.00,9.00,29.00,55.00,72.00
PhoneService,7043,2,Yes,6361,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MultipleLines,7043,3,No,3390,NaN,NaN,NaN,NaN,NaN,NaN,NaN
InternetService,7043,3,Fiber optic,3096,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OnlineSecurity,7043,3,No,3498,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Raw Data Quality Audit

Before performing transformations or feature engineering, the raw dataset is
audited for duplicates, null values, blank values, identifier uniqueness,
data types, and categorical-value consistency.

The purpose of this stage is to understand the quality of the source data
before making any cleaning decisions.

### 3.1 Duplicate Records

Duplicate records can distort customer-level churn rates and revenue
calculations, so duplicate rows are checked before analysis.

In [27]:
duplicate_count = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_count:,}")

Duplicate rows: 0


### 3.2 Missing Values

A standard null-value audit is performed using Pandas `isna()`.

This is only the first stage of the missing-value investigation because
blank strings or whitespace-only values may not be recognized as nulls.

In [28]:
null_counts = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
)

null_counts

customerID          0
DeviceProtection    0
TotalCharges        0
MonthlyCharges      0
PaymentMethod       0
PaperlessBilling    0
Contract            0
StreamingMovies     0
StreamingTV         0
TechSupport         0
OnlineBackup        0
gender              0
OnlineSecurity      0
InternetService     0
MultipleLines       0
PhoneService        0
tenure              0
Dependents          0
Partner             0
SeniorCitizen       0
Churn               0
dtype: int64

In [29]:
print(f"Total null cells: {df.isna().sum().sum():,}")

Total null cells: 0


### 3.3 Blank and Whitespace-Only Values

The raw CSV may contain blank or whitespace-only strings that are not
identified by `isna()`.

Each object/string column is therefore checked for values that are empty
after removing surrounding whitespace.

In [30]:
object_columns = df.select_dtypes(include="object").columns

blank_counts = {}

for col in object_columns:
    blank_counts[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

blank_counts = (
    pd.Series(blank_counts)
    .sort_values(ascending=False)
)

blank_counts

TotalCharges        11
customerID           0
gender               0
PaymentMethod        0
PaperlessBilling     0
Contract             0
StreamingMovies      0
StreamingTV          0
TechSupport          0
DeviceProtection     0
OnlineBackup         0
OnlineSecurity       0
InternetService      0
MultipleLines        0
PhoneService         0
Dependents           0
Partner              0
Churn                0
dtype: int64

In [31]:
blank_counts[blank_counts > 0]

TotalCharges    11
dtype: int64

### 3.4 TotalCharges Data-Type Investigation

`TotalCharges` represents cumulative charges but was loaded as an object
column rather than a numeric column.

The column is investigated for non-numeric or blank values before deciding
how it should be cleaned and converted.

In [32]:
total_charges_as_numeric = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

conversion_failures = (
    df["TotalCharges"].notna()
    & total_charges_as_numeric.isna()
)

print(f"Values that fail numeric conversion: {conversion_failures.sum():,}")

Values that fail numeric conversion: 11


In [33]:
df.loc[conversion_failures, ["customerID", "tenure", "TotalCharges", "Churn"]]

,customerID,tenure,TotalCharges,Churn
488,4472-LVYGI,0,,No
753,3115-CZMZD,0,,No
936,5709-LVOEQ,0,,No
1082,4367-NUYAO,0,,No
1340,1371-DWPAZ,0,,No
3331,7644-OMVMY,0,,No
3826,3213-VVOLG,0,,No
4380,2520-SGTTA,0,,No
5218,2923-ARZLG,0,,No
6670,4075-WKNIU,0,,No


In [34]:
df.loc[
    conversion_failures,
    ["tenure", "Churn"]
].sort_values("tenure")

,tenure,Churn
488,0,No
753,0,No
936,0,No
1082,0,No
1340,0,No
3331,0,No
3826,0,No
4380,0,No
5218,0,No
6670,0,No


### 3.5 Customer Identifier Uniqueness

Because the analysis is performed at customer level, each `customerID`
should represent one customer record.

Customer identifier uniqueness is therefore checked before downstream
aggregation and modeling.

In [35]:
unique_customers = df["customerID"].nunique()
total_rows = len(df)

print(f"Total rows: {total_rows:,}")
print(f"Unique customer IDs: {unique_customers:,}")
print(f"Duplicate customer IDs: {total_rows - unique_customers:,}")

Total rows: 7,043
Unique customer IDs: 7,043
Duplicate customer IDs: 0


### 3.6 Categorical Value Consistency

Categorical columns are reviewed for unexpected spellings, inconsistent
capitalization, or unusual categories that could affect segmentation.

In [36]:
categorical_columns = df.select_dtypes(include="object").columns

for col in categorical_columns:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- customerID ---
customerID
7590-VHVEG    1
3791-LGQCY    1
6008-NAIXK    1
5956-YHHRX    1
5365-LLFYV    1
             ..
9796-MVYXX    1
2637-FKFSY    1
1552-AAGRX    1
4304-TSPVK    1
3186-AJIEK    1
Name: count, Length: 7043, dtype: int64

--- gender ---
gender
Male      3555
Female    3488
Name: count, dtype: int64

--- Partner ---
Partner
No     3641
Yes    3402
Name: count, dtype: int64

--- Dependents ---
Dependents
No     4933
Yes    2110
Name: count, dtype: int64

--- PhoneService ---
PhoneService
Yes    6361
No      682
Name: count, dtype: int64

--- MultipleLines ---
MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64

--- InternetService ---
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

--- OnlineSecurity ---
OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64

--- OnlineBackup -

### 3.7 Raw Data Quality Summary

The initial audit produced the following observations:

- The dataset contains 7,043 customer records and 21 columns.
- No duplicate rows were identified.
- No Pandas null values were detected through `isna()`.
- A separate blank-value audit identified 11 blank values in `TotalCharges`.
- All 11 `TotalCharges` records that failed numeric conversion correspond to customers with `tenure = 0`.
- All 11 of these customers are currently classified as non-churned (`Churn = No`).
- `customerID` is unique across all 7,043 records.
- The categorical variables showed consistent category labels with no obvious spelling or capitalization inconsistencies.
- `TotalCharges` requires explicit cleaning and numeric conversion before it can be reliably used in downstream analysis.

No values are modified or removed during this raw-data audit. Cleaning decisions are deferred to the dedicated Pandas cleaning and feature-engineering stage.